# RAV-Med — Step 2: External Validation on HealthContradict

**Why this experiment.** Everything so far used PubMedQA with *programmatic* corruptions (rule-based negation, percentile-band distractors). A reviewer's first question is whether ERS also works on **naturally written, expert-labelled conflicting evidence**. HealthContradict (Zhang et al., npj Digital Medicine 2026; arXiv:2512.02299) provides exactly that: 920 pairs of real web documents for the same health question, one whose stance matches the scientific evidence (**correct**) and one that contradicts it (**incorrect**).

**What is tested (fixed before running):**
* **Primary endpoint:** within each pair, is ERS(correct) > ERS(incorrect)? (*pairwise accuracy*, 0.5 = chance, ties count 0.5.)
* **Ablation:** same endpoint for similarity-only and contradiction-only scoring. Both documents are on-topic, so similarity-only is expected to sit near chance.
* **Claim-relative check:** the endpoint is computed twice, with a **reference claim** (the evidence-supported conclusion; uses the answer) and a **label-free claim** (affirmative rewrite of the question; uses no label). The gap between them measures the claim-relative limitation on external data.
* **Controls:** a document-length-only baseline, an irrelevant-document control (documents from other questions), and transfer of the fixed thresholds T = 0.0/0.2/0.4 **without retuning**.

**Honest design notes.**
* The 920 pairs come from only **81 questions** (1–77 pairs each) and some documents repeat, so pairs are *not* independent. All intervals use a **question-level cluster bootstrap**, and a question-weighted version is reported alongside.
* Documents average ~2,000 words, far above the 256-token NLI limit. Each document is split into 80-word windows; the **primary** document score is the ERS of its **most similar window** (what a retriever would hand to a generator). Two alternatives are reported as sensitivity analyses, not as the headline.
* The model never sees the labels except through the *reference claim*.

## 1. Install

In [ ]:
!pip install -q sentence-transformers transformers accelerate bitsandbytes scikit-learn scipy

## 2. Configuration

In [ ]:
import os, re, json, random, hashlib, subprocess
import numpy as np, pandas as pd, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

DEVICE      = "cuda"
LLM_NAME    = "Qwen/Qwen2.5-7B-Instruct"     # same generator as Step 1; only ~162 short generations are needed
LOAD_4BIT   = True                            # True on a free T4; False on L4/A100
EMBED_MODEL = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"   # as in the paper
NLI_MODEL   = "roberta-large-mnli"                                        # as in the paper

CHUNK_WORDS = 80      # window size; chosen so windows fit the 256-token NLI limit
MIN_CHUNK_WORDS = 15  # drop a trailing window shorter than this (kept if it is the only one)
MAX_CHUNKS  = 120     # cap per document (= first 9,600 words; ~2% of documents are longer); the share truncated is printed
TOP_K       = 3       # for the top-3 sensitivity analysis
N_IRRELEVANT_PER_TOPIC = 5
N_BOOT      = 2000
BATCH_SIZE  = 64

## 3. Load HealthContradict and define *correct* / *incorrect*

In [ ]:
if not os.path.exists("HealthContradict"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tinaboya/HealthContradict"], check=True)

rows = [json.loads(l) for l in open("HealthContradict/dataset/dataset_ready.jsonl", encoding="utf-8")]
raw = pd.DataFrame(rows)
raw["topic_id"] = raw["topic_id"].astype(int)
# in this file doc_a always has stance "yes" and doc_b stance "no"; the evidence-supported one matches query_stance
raw["correct_text"]   = np.where(raw.query_stance == "yes", raw.doc_a, raw.doc_b)
raw["incorrect_text"] = np.where(raw.query_stance == "yes", raw.doc_b, raw.doc_a)

def did(t): return hashlib.md5(t.encode("utf-8")).hexdigest()[:12]
raw["correct_id"]   = raw.correct_text.map(did)
raw["incorrect_id"] = raw.incorrect_text.map(did)
docs = {}
for col in ["correct", "incorrect"]:
    for i, t in zip(raw[f"{col}_id"], raw[f"{col}_text"]): docs[i] = t
pairs = raw[["instance_id", "topic_id", "query", "query_stance", "correct_id", "incorrect_id"]].copy()
topics = pairs.drop_duplicates("topic_id").set_index("topic_id")[["query", "query_stance"]]

print(f"{len(pairs)} pairs | {len(topics)} questions | {len(docs)} unique documents")
print("pairs per question:", pairs.groupby("topic_id").size().describe().round(1).to_dict())
print("question answer (stance):", topics.query_stance.value_counts().to_dict())
L = lambda ids: pd.Series([len(docs[i].split()) for i in ids])
print("words per doc  correct/incorrect (mean):", round(L(pairs.correct_id).mean()), round(L(pairs.incorrect_id).mean()))

## 4. Build the two claims for each of the 81 questions
* **Reference claim** — given the question *and* the evidence-supported answer, one declarative sentence stating the supported conclusion (the claim-given setting RAV-Med is defined for).
* **Label-free claim** — an affirmative rewrite of the question that never sees the answer (the deployable setting).

All 81 pairs of claims are printed so you can check them by eye; they are also saved to `hc_claims.csv`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
tok = AutoTokenizer.from_pretrained(LLM_NAME, padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token
if LOAD_4BIT:
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, device_map={"": 0}, quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")).eval()
else:
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(DEVICE).eval()

SYS = "You rewrite health questions as single declarative sentences. Output only the sentence."
def p_free(q):
    return ("Rewrite this health question or search topic as ONE affirmative declarative sentence that asserts "
            "the proposition being asked about (for example, 'Melatonin reduces jet lag.'). Do not answer it, "
            f"hedge, or add information.\n\nQuestion: {q}\nSentence:")
def p_ref(q, stance):
    return ("Write ONE declarative sentence stating the conclusion that is supported by the scientific evidence, "
            f"given the question and the evidence-based answer. If the answer is No, state that the proposition does not hold. "
            "Do not hedge and do not use the words 'yes' or 'no'.\n\n"
            f"Question: {q}\nEvidence-based answer: {'Yes' if stance == 'yes' else 'No'}\nSentence:")

@torch.no_grad()
def gen(prompts, bs=8):
    out = []
    for i in range(0, len(prompts), bs):
        texts = [tok.apply_chat_template([{"role": "system", "content": SYS}, {"role": "user", "content": p}],
                 tokenize=False, add_generation_prompt=True) for p in prompts[i:i+bs]]
        enc = tok(texts, return_tensors="pt", padding=True).to(DEVICE)
        o = llm.generate(**enc, max_new_tokens=60, do_sample=False, pad_token_id=tok.pad_token_id)
        out += [d.strip().split("\n")[0].strip().strip('"') for d in tok.batch_decode(o[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return out

claims = topics.copy()
claims["claim_free"] = gen([p_free(q) for q in claims["query"]])
claims["claim_ref"]  = gen([p_ref(q, s) for q, s in zip(claims["query"], claims["query_stance"])])
claims.to_csv("hc_claims.csv")
pd.set_option("display.max_colwidth", 140, "display.width", 250, "display.max_rows", 200)
bad = claims[(claims.claim_free.str.len() < 15) | (claims.claim_ref.str.len() < 15) | claims.claim_ref.str.lower().str.contains(r"\b(yes|no)\b")]
print("Automatic flags (empty / too short / contains yes-no):", len(bad)); display(bad)
for t, r in claims.iterrows():
    print(f"[{t}] ({r.query_stance}) {r['query']}\n   FREE: {r.claim_free}\n   REF : {r.claim_ref}")
del llm; import gc; gc.collect(); torch.cuda.empty_cache()

> **Read the printout above before continuing.** Check that every REF claim states the *evidence-supported* direction (answer *no* → a negated claim) and that every FREE claim is affirmative. If a few are wrong, edit them in `hc_claims.csv`, re-upload, and re-run from section 5; report how many you changed.

## 5. Chunk the documents and load the scoring models

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print("SBERT encoder max_seq_length:", embedder.max_seq_length, "tokens  <- also the setting used in the PubMedQA runs (never changed there)")
nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL)
from transformers import AutoModelForSequenceClassification
nli = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(DEVICE).eval()
c_idx = [i for i, l in nli.config.id2label.items() if l.lower().startswith("contra")][0]

def chunk(text):
    w = text.split()
    ch = [" ".join(w[i:i+CHUNK_WORDS]) for i in range(0, len(w), CHUNK_WORDS)]
    if len(ch) > 1 and len(ch[-1].split()) < MIN_CHUNK_WORDS: ch = ch[:-1]
    return ch[:MAX_CHUNKS] if ch else [text]

# irrelevant-document control: for each question, documents drawn from OTHER questions
all_by_topic = {}
for _, r in pairs.iterrows():
    all_by_topic.setdefault(r.topic_id, set()).update([r.correct_id, r.incorrect_id])
irr_rows = []
for t in topics.index:
    other = sorted(set().union(*[v for k, v in all_by_topic.items() if k != t]) - all_by_topic[t])
    for d in rng.choice(other, size=N_IRRELEVANT_PER_TOPIC, replace=False): irr_rows.append((t, d))
irr = pd.DataFrame(irr_rows, columns=["topic_id", "doc_id"])

need = pd.concat([pairs[["topic_id"]].assign(doc_id=pairs.correct_id), pairs[["topic_id"]].assign(doc_id=pairs.incorrect_id), irr]).drop_duplicates()
doc_ids = sorted(need.doc_id.unique())
chunk_rows = []
trunc = 0
for d in doc_ids:
    n_words = len(docs[d].split())
    if n_words > MAX_CHUNKS * CHUNK_WORDS: trunc += 1
    for k, c in enumerate(chunk(docs[d])): chunk_rows.append((d, k, c))
chunks = pd.DataFrame(chunk_rows, columns=["doc_id", "k", "text"])
print(f"{len(doc_ids)} documents -> {len(chunks)} windows | documents truncated at {MAX_CHUNKS} windows: {trunc} ({trunc/len(doc_ids):.1%})")
sample = chunks.text.sample(min(2000, len(chunks)), random_state=SEED).tolist()
n_sb = np.array([len(embedder.tokenizer.tokenize(t)) for t in sample]); n_nli = np.array([len(nli_tok.tokenize(t)) for t in sample])
print(f"windows longer than SBERT limit: {(n_sb > embedder.max_seq_length).mean():.1%} | longer than the 256-token NLI limit: {(n_nli > 250).mean():.1%}")

## 6. Score every (question, document) against both claims

In [ ]:
chunk_emb = embedder.encode(chunks.text.tolist(), batch_size=BATCH_SIZE, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True)

@torch.no_grad()
def contra(prem, hyp, bs=BATCH_SIZE):
    order = np.argsort([len(p) for p in prem]); out = np.zeros(len(prem))
    for i in range(0, len(prem), bs):
        idx = order[i:i+bs]
        enc = nli_tok([prem[j] for j in idx], [hyp[j] for j in idx], truncation=True, padding=True, max_length=256, return_tensors="pt").to(DEVICE)
        out[idx] = torch.softmax(nli(**enc).logits, -1)[:, c_idx].cpu().numpy()
    return out

chunk_index = chunks.reset_index().groupby("doc_id")["index"].apply(list).to_dict()
claim_emb = {(t, ct): embedder.encode([claims.loc[t, f"claim_{ct}"]], normalize_embeddings=True, convert_to_numpy=True)[0]
             for t in claims.index for ct in ("ref", "free")}

def score_pairs(need_df, claim_type):
    """Return one row per (topic, doc) with S/C aggregated three ways."""
    recs, prem, hyp, meta = [], [], [], []
    for t, d in need_df.itertuples(index=False):
        ids = chunk_index[d]
        S = chunk_emb[ids] @ claim_emb[(t, claim_type)]
        for j, s in zip(ids, S): meta.append((t, d, s)); prem.append(chunks.text[j]); hyp.append(claims.loc[t, f"claim_{claim_type}"])
    Cs = contra(prem, hyp)
    m = pd.DataFrame(meta, columns=["topic_id", "doc_id", "S"]); m["C"] = Cs
    out = []
    for (t, d), g in m.groupby(["topic_id", "doc_id"], sort=False):
        g = g.sort_values("S", ascending=False)
        top = g.iloc[0]; topk = g.iloc[:TOP_K]
        out.append(dict(topic_id=t, doc_id=d, claim_type=claim_type,
                        S_top1=top.S, C_top1=top.C, ERS_top1=top.S - top.C,
                        ERS_topk=(topk.S - topk.C).mean(), ERS_maxC=g.S.max() - g.C.max(), n_windows=len(g)))
    return pd.DataFrame(out)

scores = pd.concat([score_pairs(need, ct) for ct in ("ref", "free")], ignore_index=True)
scores.to_csv("hc_doc_scores.csv", index=False)
print(scores.groupby("claim_type")[["S_top1", "C_top1", "ERS_top1"]].mean().round(3))

## 7. Primary analysis — pairwise accuracy within each pair
Cluster bootstrap resamples **questions** (81), carrying all of a question's pairs with it. The *question-weighted* version first averages within each question, so a question with 77 pairs counts the same as one with 1.

In [ ]:
from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score
S_ = scores.set_index(["topic_id", "doc_id", "claim_type"])

def pair_scores(claim_type, scorer):
    c = S_.loc[list(zip(pairs.topic_id, pairs.correct_id, [claim_type] * len(pairs)))]
    i = S_.loc[list(zip(pairs.topic_id, pairs.incorrect_id, [claim_type] * len(pairs)))]
    if scorer == "similarity": return c.S_top1.values, i.S_top1.values
    if scorer == "contradiction": return -c.C_top1.values, -i.C_top1.values
    return c[scorer].values, i[scorer].values        # ERS_top1 / ERS_topk / ERS_maxC

def win(a, b): return (a > b).astype(float) + 0.5 * (a == b)
tid = pairs.topic_id.values; utid = np.unique(tid); tpos = {t: np.where(tid == t)[0] for t in utid}
boot_t = rng.integers(0, len(utid), size=(N_BOOT, len(utid)))

def summarize(w):
    per_t = np.array([w[tpos[t]].mean() for t in utid]); n_t = np.array([len(tpos[t]) for t in utid]); s_t = np.array([w[tpos[t]].sum() for t in utid])
    pooled = w.mean(); bp = (s_t[boot_t].sum(1) / n_t[boot_t].sum(1))
    macro = per_t.mean(); bm = per_t[boot_t].mean(1)
    return dict(pooled=pooled, pooled_lo=np.percentile(bp, 2.5), pooled_hi=np.percentile(bp, 97.5),
                q_weighted=macro, qw_lo=np.percentile(bm, 2.5), qw_hi=np.percentile(bm, 97.5)), per_t

rows, per_question = [], {}
for ct, ctname in [("ref", "reference claim"), ("free", "label-free claim")]:
    for scorer, sname in [("similarity", "Similarity only"), ("contradiction", "Contradiction only"), ("ERS_top1", "Full ERS (primary: top-1 window)"),
                          ("ERS_topk", "ERS, mean of top-3 windows"), ("ERS_maxC", "ERS, max-S minus max-C")]:
        a, b = pair_scores(ct, scorer); w = win(a, b); s, per_t = summarize(w)
        per_question[(ct, scorer)] = per_t
        rows.append(dict(claim=ctname, scorer=sname, **s))
T_MAIN = pd.DataFrame(rows)
# length-only baseline (log words), no model involved
lc, li = np.log(L(pairs.correct_id).values), np.log(L(pairs.incorrect_id).values); s, _ = summarize(win(lc, li))
T_MAIN = pd.concat([T_MAIN, pd.DataFrame([dict(claim="none", scorer="Length only (log words)", **s)])], ignore_index=True)
print(T_MAIN.round(3).to_string(index=False)); T_MAIN.to_csv("hc_table_pairwise_accuracy.csv", index=False)

print("\nQuestion-level tests (n = 81 questions)")
def wtest(x, y=None):
    return wilcoxon(x - (0.5 if y is None else y), alternative="two-sided").pvalue
tests = [("ERS (ref claim) vs chance 0.5", wtest(per_question[("ref", "ERS_top1")])),
         ("Similarity-only (ref claim) vs chance 0.5", wtest(per_question[("ref", "similarity")])),
         ("ERS vs similarity-only (ref claim, paired)", wtest(per_question[("ref", "ERS_top1")], per_question[("ref", "similarity")])),
         ("ERS (label-free claim) vs chance 0.5", wtest(per_question[("free", "ERS_top1")]))]
for n_, p in tests: print(f"  {n_}: Wilcoxon p = {p:.4g}")

## 8. Claim-relative check — does it depend on the question's answer?

In [ ]:
st = pairs.query_stance.values
rows = []
for ct, ctname in [("ref", "reference claim"), ("free", "label-free claim")]:
    for stance in ["yes", "no"]:
        m = st == stance
        for scorer, sname in [("similarity", "Similarity only"), ("ERS_top1", "Full ERS")]:
            a, b = pair_scores(ct, scorer); w = win(a, b)[m]
            tt = np.unique(tid[m]); pos = [np.where((tid == t) & m)[0] for t in tt]
            sums = np.array([win(a, b)[p].sum() for p in pos]); ns = np.array([len(p) for p in pos]); bi = rng.integers(0, len(tt), size=(N_BOOT, len(tt)))
            bs = sums[bi].sum(1) / ns[bi].sum(1)
            rows.append(dict(claim=ctname, gold_answer=stance, scorer=sname, n_pairs=int(m.sum()), n_questions=len(tt),
                             pairwise_acc=w.mean(), lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5)))
T_STANCE = pd.DataFrame(rows); print(T_STANCE.round(3).to_string(index=False)); T_STANCE.to_csv("hc_table_by_answer.csv", index=False)

## 9. Transfer of the fixed thresholds (no retuning) and the irrelevant-document control

In [ ]:
rows = []
for ct, ctname in [("ref", "reference claim"), ("free", "label-free claim")]:
    c, i = pair_scores(ct, "ERS_top1")
    for T in [0.0, 0.2, 0.4]:
        wc = (c >= T).astype(float); wi = (i >= T).astype(float)
        rows.append(dict(claim=ctname, T=T, correct_doc_SAFE=wc.mean(), incorrect_doc_SAFE=wi.mean(),
                         correct_doc_UNSAFE=(c < T - 0.2).mean(), incorrect_doc_UNSAFE=(i < T - 0.2).mean()))
T_THR = pd.DataFrame(rows); print(T_THR.round(3).to_string(index=False)); T_THR.to_csv("hc_table_threshold_transfer.csv", index=False)

# irrelevant-document control: correct documents vs documents from other questions (AUROC, correct = positive)
rows = []
cor = pairs[["topic_id", "correct_id"]].drop_duplicates().rename(columns={"correct_id": "doc_id"})
for ct, ctname in [("ref", "reference claim"), ("free", "label-free claim")]:
    for scorer, sname in [("S_top1", "Similarity only"), ("C_top1", "Contradiction only"), ("ERS_top1", "Full ERS")]:
        g1 = cor.merge(scores[scores.claim_type == ct], on=["topic_id", "doc_id"]); g0 = irr.merge(scores[scores.claim_type == ct], on=["topic_id", "doc_id"])
        sign = -1 if scorer == "C_top1" else 1
        y = np.r_[np.ones(len(g1)), np.zeros(len(g0))]; x = np.r_[sign * g1[scorer], sign * g0[scorer]]; tq = np.r_[g1.topic_id, g0.topic_id]
        auc = roc_auc_score(y, x); bs = []
        for b in range(1000):
            pick = rng.choice(utid, size=len(utid), replace=True); idx = np.concatenate([np.where(tq == t)[0] for t in pick])
            if len(np.unique(y[idx])) == 2: bs.append(roc_auc_score(y[idx], x[idx]))
        rows.append(dict(claim=ctname, scorer=sname, AUROC_correct_vs_irrelevant=auc, lo=np.percentile(bs, 2.5), hi=np.percentile(bs, 97.5)))
T_IRR = pd.DataFrame(rows); print(T_IRR.round(3).to_string(index=False)); T_IRR.to_csv("hc_table_irrelevant_control.csv", index=False)

## 10. Figure and save everything

In [ ]:
import matplotlib.pyplot as plt
t = T_MAIN[T_MAIN.scorer.isin(["Similarity only", "Contradiction only", "Full ERS (primary: top-1 window)"])]
fig, ax = plt.subplots(figsize=(7, 4)); w = 0.35
for k, (cl, off) in enumerate([("reference claim", -w/2), ("label-free claim", w/2)]):
    s = t[t.claim == cl]; x = np.arange(len(s)) + off
    ax.bar(x, s.pooled, w, yerr=[s.pooled - s.pooled_lo, s.pooled_hi - s.pooled], capsize=3, label=cl)
ax.axhline(0.5, color="black", lw=0.8, ls="--"); ax.set_xticks(range(3)); ax.set_xticklabels(["Similarity\nonly", "Contradiction\nonly", "Full ERS"])
ax.set_ylabel("Pairwise accuracy (correct > incorrect)"); ax.set_ylim(0, 1); ax.legend(); ax.set_title("HealthContradict: 920 pairs, 81 questions (cluster-bootstrap 95% CI)")
plt.tight_layout(); plt.savefig("hc_fig_pairwise_accuracy.png", dpi=150); plt.show()

import shutil
os.makedirs("rav_med_step2_outputs", exist_ok=True)
for f in ["hc_claims.csv", "hc_doc_scores.csv", "hc_table_pairwise_accuracy.csv", "hc_table_by_answer.csv", "hc_table_threshold_transfer.csv",
          "hc_table_irrelevant_control.csv", "hc_fig_pairwise_accuracy.png"]: shutil.copy(f, "rav_med_step2_outputs")
shutil.make_archive("rav_med_step2_outputs", "zip", "rav_med_step2_outputs")
try:
    from google.colab import files; files.download("rav_med_step2_outputs.zip")
except ImportError: pass
print("Saved rav_med_step2_outputs.zip")